In [0]:
%sql
USE CATALOG dbacademy;
USE SCHEMA default;

In [0]:
spark.conf.set('spark.databricks.io.cache.enabled', False)

## Data Creation

In [0]:
from pyspark.sql.functions import *

transactions_df = (spark
                   .range(0, 150000000, 1, 32)
                   .select(
                     'id',
                     round(rand()*10000, 2).alias('amount'),
                     (col('id') % 10).alias('country_id'),
                     (col('id')% 100).alias('store_id')
                   )
                  )
transactions_df.display()

In [0]:
transactions_df.write.mode('overwrite').saveAsTable('transactions')

In [0]:
stores_df = (spark
             .range(0, 99)
             .select(
                 'id',
                 round(rand() * 100, 0).alias('employees'),
                 (col('id') % 10).alias('country_id'),
                 expr('uuid()').alias('name')
             )
)
stores_df.display()

In [0]:
stores_df.write.saveAsTable('stores')

In [0]:
countries = [
    (0, "Italy"),
    (1, "Canada"),
    (2, "Mexico"),
    (3, "China"),
    (4, "Germany"),
    (5, "UK"),
    (6, "Japan"),
    (7, "Korea"),
    (8, "Australia"),
    (9, "France"),
    (10, "Spain"),
    (11, "USA")
]

columns = ['id', 'name']
countries_df = spark.createDataFrame(data=countries, schema=columns)
countries_df.display()

In [0]:
countries_df.write.saveAsTable('countries')

## Joins

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.databricks.adaptive.autoBroadcastJoinThreshold", -1)

joined_df = spark.sql("""
  SELECT
    transactions.id,
    amount,
    countries.name as country_name,
    employees,
    stores.name as store_name
  FROM transactions
  LEFT JOIN 
    stores
    ON transactions.store_id = stores.id
  LEFT JOIN 
    countries
    ON transactions.country_id = countries.id                      
""")

joined_df.write.mode('overwrite').saveAsTable('transact_countries')


## Broadcast Join

In [0]:
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.databricks.adaptive.autoBroadcastJoinThreshold")

joined_df = spark.sql("""
  SELECT
    transactions.id,
    amount,
    countries.name as country_name,
    employees,
    stores.name as store_name
  FROM transactions
  LEFT JOIN 
    stores
    ON transactions.store_id = stores.id
  LEFT JOIN 
    countries
    ON transactions.country_id = countries.id                      
""")

joined_df.write.mode('overwrite').saveAsTable('transact_countries')

## Aggregations

In [0]:
%sql
SELECT
  country_id,
  COUNT(*) AS count,
  AVG(amount) AS avg_amount
  FROM transactions
  GROUP BY country_id
  ORDER BY count DESC